# Daily strikeout projections

Frozen LightGBM k-rate × Ridge TBF → `expected_K` + fair American odds (**3.5–8.5**).

### What is scraped from where?

| Piece | Source |
|---|---|
| Batting orders (1–9) | **RotoGrinders only** |
| Starting pitcher (dual) | RG card **and** MLB schedule probable |

Dual rows swap the **SP identity** only. Both rows use the **same RG opponent lineup** aggregates (`opp_lineup_*`). There is no separate MLB batting-order scrape yet — so lineup risk is shared; SP disagreement is what you see as stacked rows.

Overnight, RG may still show yesterday’s SPs while MLB shows today’s probables. Prefer `mlb_probable` / `is_preferred` until RG refreshes.

Scoring runs in a fresh subprocess (avoids Jupyter/Windows LightGBM AVs). No parquet clutter from this notebook — use `production/log_projections.py` to persist.

In [1]:
from __future__ import annotations

import pickle
import subprocess
import sys
import tempfile
from datetime import date
from pathlib import Path

import polars as pl

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "Python").exists():
    if (ROOT.parent / "src" / "Python").exists():
        ROOT = ROOT.parent.resolve()
    else:
        raise FileNotFoundError(f"Cannot find src/Python from cwd={Path.cwd()}")

WORKER = ROOT / "production" / "_notebook_score.py"
ALLOW_STALE = True
SLATE_DATE: date | None = None
QUIET_WARNINGS = True

print("repo:", ROOT)
print("python:", sys.executable)

repo: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props
python: c:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\.venv\Scripts\python.exe


## 1. Score slate

In [2]:
with tempfile.NamedTemporaryFile(suffix=".pkl", delete=False) as tmp:
    out_path = Path(tmp.name)

cmd = [sys.executable, str(WORKER), str(out_path)]
if ALLOW_STALE:
    cmd.append("--allow-stale")
if SLATE_DATE is not None:
    cmd.extend(["--date", SLATE_DATE.isoformat()])
if QUIET_WARNINGS:
    cmd.append("--quiet")

proc = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
if proc.returncode != 0:
    raise RuntimeError(
        "Board scorer failed.\n"
        f"cmd: {' '.join(cmd)}\nstdout:\n{proc.stdout}\nstderr:\n{proc.stderr}"
    )

payload = pickle.loads(out_path.read_bytes())
out_path.unlink(missing_ok=True)

board: pl.DataFrame = payload["board"]
preferred: pl.DataFrame = payload["preferred"]
build_meta = payload["build_meta"]
report = payload["report"]

VIEW = [
    c
    for c in (
        "away_team",
        "home_team",
        "is_home",
        "player_name",
        "starter_source",
        "is_preferred",
        "starter_disagreement",
        "expected_K",
        "projected_tbf",
        "k_rate_pred",
        "opp_lineup_k",
        "opp_lineup_size",
        "fair_amer_3_5",
        "fair_amer_4_5",
        "fair_amer_5_5",
        "fair_amer_6_5",
        "fair_amer_7_5",
        "fair_amer_8_5",
        "p_over_3_5",
        "p_over_4_5",
        "p_over_5_5",
        "p_over_6_5",
        "p_over_7_5",
        "p_over_8_5",
    )
    if c in board.columns
]


def _fmt(df: pl.DataFrame) -> pl.DataFrame:
    cols = [c for c in VIEW if c in df.columns]
    out = df.select(cols)
    exprs = []
    if "expected_K" in out.columns:
        exprs.append(pl.col("expected_K").round(2))
    if "projected_tbf" in out.columns:
        exprs.append(pl.col("projected_tbf").round(2))
    if "k_rate_pred" in out.columns:
        exprs.append(pl.col("k_rate_pred").round(3))
    if "opp_lineup_k" in out.columns:
        exprs.append(pl.col("opp_lineup_k").round(3))
    exprs.extend(pl.col(c).round(3) for c in cols if c.startswith("p_over_"))
    return out.with_columns(exprs) if exprs else out


def show_scrollable(df: pl.DataFrame, height: int = 520):
    """Fixed-height scrollable HTML table (sticky header)."""
    from IPython.display import HTML, display

    pdf = df.to_pandas()
    # Compact SP-focused columns first when present.
    front = [
        c
        for c in (
            "player_name",
            "away_team",
            "home_team",
            "is_home",
            "starter_source",
            "expected_K",
            "projected_tbf",
            "k_rate_pred",
        )
        if c in pdf.columns
    ]
    rest = [c for c in pdf.columns if c not in front]
    pdf = pdf[front + rest]
    table = pdf.to_html(index=False, classes="proj-board")
    display(
        HTML(
            f"""
<style>
  .proj-scroll {{
    max-height: {height}px;
    overflow: auto;
    border: 1px solid #4443;
    border-radius: 6px;
  }}
  .proj-scroll table.proj-board {{
    border-collapse: collapse;
    width: max-content;
    min-width: 100%;
    font-size: 13px;
  }}
  .proj-scroll thead th {{
    position: sticky;
    top: 0;
    background: var(--jp-layout-color1, #1e1e1e);
    z-index: 1;
    text-align: left;
    padding: 6px 10px;
    white-space: nowrap;
  }}
  .proj-scroll tbody td {{
    text-align: left;
    padding: 4px 10px;
    white-space: nowrap;
  }}
</style>
<div class="proj-scroll">{table}</div>
"""
        )
    )


print("slate_date:", build_meta.get("slate_date"))
print("rolling_max:", build_meta.get("rolling_max_date"), "stale_days:", build_meta.get("stale_days"))
print("rows:", board.height, "| disagreement rows:", build_meta.get("n_disagreement_rows"))
print("mean expected_K:", round(float(report["mean_expected_K"]), 3))
print("lineup_source: RotoGrinders batting orders for all rows (SP dual only)")

slate_date: 2026-07-28
rolling_max: 2026-07-26 stale_days: 2
rows: 32 | disagreement rows: 0
mean expected_K: 4.946
lineup_source: RotoGrinders batting orders for all rows (SP dual only)


## 2. Both sources (stacked)

All dual rows. Same `opp_lineup_*` for a game/side — only `player_name` / SP form / TBF change by source.

In [3]:
both = _fmt(board)
show_scrollable(both)

player_name,away_team,home_team,is_home,starter_source,expected_K,projected_tbf,k_rate_pred,is_preferred,starter_disagreement,opp_lineup_k,opp_lineup_size,fair_amer_3_5,fair_amer_4_5,fair_amer_5_5,fair_amer_6_5,fair_amer_7_5,fair_amer_8_5,p_over_3_5,p_over_4_5,p_over_5_5,p_over_6_5,p_over_7_5,p_over_8_5
Shane Bieber,TOR,WSH,False,rotogrinders,4.63,22.34,0.207,True,False,0.204,9,-230,104,237,554,1390,3891,0.697,0.491,0.297,0.153,0.067,0.025
Cade Cavalli,TOR,WSH,True,rotogrinders,5.11,22.84,0.224,True,False,0.196,9,-376,-156,143,315,724,1803,0.790,0.609,0.412,0.241,0.121,0.053
Cal Quantrill,TEX,TB,False,rotogrinders,3.22,19.37,0.166,True,False,0.193,9,155,405,1120,3486,12683,55021,0.392,0.198,0.082,0.028,0.008,0.002
Griffin Jax,TEX,TB,True,rotogrinders,4.90,21.23,0.231,True,False,0.210,9,-299,-124,183,418,1015,2725,0.749,0.553,0.353,0.193,0.090,0.035
Colin Rea,CHC,STL,False,rotogrinders,4.15,22.46,0.185,True,False,0.199,9,-150,161,381,951,2616,8185,0.599,0.384,0.208,0.095,0.037,0.012
Michael McGreevy,CHC,STL,True,rotogrinders,4.10,23.26,0.176,True,False,0.207,9,-148,162,383,950,2591,8008,0.596,0.381,0.207,0.095,0.037,0.012
Logan Henderson,MIL,SF,False,rotogrinders,5.25,21.22,0.247,True,False,0.230,9,-402,-165,137,304,707,1789,0.801,0.622,0.422,0.247,0.124,0.053
Landen Roupp,MIL,SF,True,rotogrinders,5.30,23.06,0.230,True,False,0.219,9,-425,-175,127,278,628,1528,0.809,0.637,0.440,0.265,0.137,0.061
Michael Lorenzen,COL,SD,False,rotogrinders,4.03,22.56,0.179,True,False,0.205,9,-156,153,360,884,2380,7246,0.609,0.395,0.218,0.102,0.040,0.014
Mike King,COL,SD,True,rotogrinders,5.10,22.99,0.222,True,False,0.219,9,-361,-150,149,329,758,1902,0.783,0.600,0.402,0.233,0.117,0.050


## 3. RotoGrinders starters only

`starter_source == "rotogrinders"`. Early card / DFS names; may lag overnight.

In [4]:
rg_only = _fmt(board.filter(pl.col("starter_source") == "rotogrinders"))
rg_only

away_team,home_team,is_home,player_name,starter_source,is_preferred,starter_disagreement,expected_K,projected_tbf,k_rate_pred,opp_lineup_k,opp_lineup_size,fair_amer_3_5,fair_amer_4_5,fair_amer_5_5,fair_amer_6_5,fair_amer_7_5,fair_amer_8_5,p_over_3_5,p_over_4_5,p_over_5_5,p_over_6_5,p_over_7_5,p_over_8_5
str,str,bool,str,str,bool,bool,f64,f64,f64,f64,u32,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64
"""TOR""","""WSH""",false,"""Shane Bieber""","""rotogrinders""",true,false,4.63,22.34,0.207,0.204,9,-230,104,237,554,1390,3891,0.697,0.491,0.297,0.153,0.067,0.025
"""TOR""","""WSH""",true,"""Cade Cavalli""","""rotogrinders""",true,false,5.11,22.84,0.224,0.196,9,-376,-156,143,315,724,1803,0.79,0.609,0.412,0.241,0.121,0.053
"""TEX""","""TB""",false,"""Cal Quantrill""","""rotogrinders""",true,false,3.22,19.37,0.166,0.193,9,155,405,1120,3486,12683,55021,0.392,0.198,0.082,0.028,0.008,0.002
"""TEX""","""TB""",true,"""Griffin Jax""","""rotogrinders""",true,false,4.9,21.23,0.231,0.21,9,-299,-124,183,418,1015,2725,0.749,0.553,0.353,0.193,0.09,0.035
"""CHC""","""STL""",false,"""Colin Rea""","""rotogrinders""",true,false,4.15,22.46,0.185,0.199,9,-150,161,381,951,2616,8185,0.599,0.384,0.208,0.095,0.037,0.012
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""CLE""","""CIN""",true,"""Chase Burns""","""rotogrinders""",true,false,6.5,23.14,0.281,0.184,9,-1171,-444,-194,109,228,487,0.921,0.816,0.66,0.478,0.305,0.17
"""NYY""","""CWS""",false,"""Gerrit Cole""","""rotogrinders""",true,false,5.77,23.1,0.25,0.231,9,-625,-251,-113,190,413,947,0.862,0.715,0.53,0.345,0.195,0.096
"""NYY""","""CWS""",true,"""Anthony Kay""","""rotogrinders""",true,false,5.11,21.93,0.233,0.227,9,-376,-155,145,321,744,1879,0.79,0.608,0.409,0.238,0.118,0.051


## 4. MLB probable starters only

`starter_source == "mlb_probable"`.

**Note:** this filter only has rows when RG and MLB **disagree** (that is when we emit an extra MLB row). When they agree, the single row is tagged `rotogrinders` with `is_preferred=True` — so it will **not** appear here. For a complete “use this SP” board, see §5.

In [5]:
mlb_only = _fmt(board.filter(pl.col("starter_source") == "mlb_probable"))
show_scrollable(mlb_only)

player_name,away_team,home_team,is_home,starter_source,expected_K,projected_tbf,k_rate_pred,is_preferred,starter_disagreement,opp_lineup_k,opp_lineup_size,fair_amer_3_5,fair_amer_4_5,fair_amer_5_5,fair_amer_6_5,fair_amer_7_5,fair_amer_8_5,p_over_3_5,p_over_4_5,p_over_5_5,p_over_6_5,p_over_7_5,p_over_8_5


## 5. Preferred board (`is_preferred`)

One row per team-side: MLB probable on disagreement, otherwise the single agreed starter. Best default for logging / betting shortlist.

In [6]:
pref_cols = [
    "player_name",
    "away_team",
    "home_team",
    "expected_K",
    "fair_amer_3_5",
    "fair_amer_4_5",
    "fair_amer_5_5",
    "fair_amer_6_5",
    "fair_amer_7_5",
    "fair_amer_8_5",
]
_pref = _fmt(preferred)
preferred_view = _pref.select([c for c in pref_cols if c in _pref.columns]).sort(
    "expected_K", descending=True
)
show_scrollable(preferred_view, height=560)

player_name,away_team,home_team,expected_K,fair_amer_3_5,fair_amer_4_5,fair_amer_5_5,fair_amer_6_5,fair_amer_7_5,fair_amer_8_5
Taj Bradley,KC,MIN,6.75,-1473,-548,-238,-112,183,381
Chris Sale,ATL,NYM,6.61,-1374,-513,-222,-105,198,419
Chase Burns,CLE,CIN,6.50,-1171,-444,-194,109,228,487
Gavin Williams,CLE,CIN,6.32,-1067,-409,-180,118,247,533
Gage Jump,BOS,ATH,6.02,-725,-286,-127,169,365,831
Gerrit Cole,NYY,CWS,5.77,-625,-251,-113,190,413,947
Reid Detmers,HOU,LAA,5.65,-540,-219,102,219,482,1128
Christian Scott,ATL,NYM,5.50,-552,-223,100,218,482,1141
Troy Melton,BAL,DET,5.37,-460,-189,118,256,573,1377
Landen Roupp,MIL,SF,5.30,-425,-175,127,278,628,1528


## 6. Persist + grade (optional)

```powershell
python production/log_projections.py --allow-stale
python production/grade_projections.py --preferred-only   # after Level 1 has yesterday
```

Log path: `artifacts/projection_log/projections.parquet`

In [7]:
{
    k: report[k]
    for k in (
        "k_rate_model",
        "k_rate_sha256",
        "tbf_model",
        "tbf_sha256",
        "tbf_alpha",
        "lines",
        "mean_expected_K",
        "approved_utc",
    )
    if k in report
}

{'k_rate_model': 'C:\\Users\\ckaplinger\\Downloads\\Personal-Projects\\MLB-Props\\artifacts\\models\\lightgbm_krate_20260728_033241.txt',
 'k_rate_sha256': '5c8e1276d1d8af4e2717c2b9d07d9c8bcf3e9c6ca358b90923e850fa542e7e86',
 'tbf_model': 'C:\\Users\\ckaplinger\\Downloads\\Personal-Projects\\MLB-Props\\artifacts\\models\\tbf_pa_ridge_workload_context_bullpen_20260728_035607.joblib',
 'tbf_sha256': '5ee23260ae5f0c0ca55436eeb5248a1ea833e1a15fb1c2dfe80d9e0244cf95d3',
 'tbf_alpha': 123.28467394420659,
 'lines': [3.5, 4.5, 5.5, 6.5, 7.5, 8.5],
 'mean_expected_K': 4.946367825952454,
 'approved_utc': '2026-07-28T14:09:27Z'}